# Module 9b: exercises  (solutions)

Each exercise asks you to compute, by hand, the numbers a chart is drawn from. The checks compare your answers with NumPy, SciPy, pandas, matplotlib and statsmodels.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Bin rules.** Write `bin_counts_by_rule(x)` returning a dict `{"sturges": k1, "freedman_diaconis": k2}` with the **number of bins** from each rule: Sturges $k=\lceil\log_2 n\rceil+1$; Freedman–Diaconis width
$2\,\text{IQR}\,n^{-1/3}$, giving $\lceil(\max-\min)/\text{width}\rceil$ bins (IQR from `np.percentile`). Apply it to `response_minutes` as `rules`.

In [ ]:
import math

import matplotlib.cbook as cbook
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

students = pd.read_csv(DATA / "students.csv")
desk = pd.read_csv(DATA / "service_desk.csv")
trial = pd.read_csv(DATA / "app_trial.csv")
rides = pd.read_csv(DATA / "daily_rides.csv", parse_dates=["date"])

def bin_counts_by_rule(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    width = 2 * iqr * n ** (-1 / 3)
    return {"sturges": math.ceil(math.log2(n)) + 1,
            "freedman_diaconis": math.ceil((x.max() - x.min()) / width)}

rules = bin_counts_by_rule(desk["response_minutes"])

In [ ]:
for data in (students["exam_score"], desk["response_minutes"], desk["call_minutes"]):
    mine = bin_counts_by_rule(data)
    assert mine["sturges"] == len(np.histogram_bin_edges(data, "sturges")) - 1
    assert mine["freedman_diaconis"] == len(np.histogram_bin_edges(data, "fd")) - 1
assert rules == {"sturges": 10, "freedman_diaconis": 27}, rules
print("✅ Correct!", rules, "(the skewed variable needs far more bins under Freedman–Diaconis)")

## Exercise 2

**ECDF and the KS distance.** Write `ecdf(x)` returning `(xs, ys)` (sorted values and $i/n$) and `ks_distance(a, b)` returning the largest absolute vertical gap between the two samples' ECDFs
(evaluate both ECDFs at every pooled data point: $\hat F(t)=\#\{x_i\le t\}/n$, e.g. via `np.searchsorted(np.sort(x), t, side="right") / n`).
Apply it to gamified vs control `minutes_active` as `d`.

In [ ]:
def ecdf(x):
    xs = np.sort(np.asarray(x, dtype=float))
    return xs, np.arange(1, len(xs) + 1) / len(xs)

def ks_distance(a, b):
    a, b = np.sort(np.asarray(a, dtype=float)), np.sort(np.asarray(b, dtype=float))
    grid = np.sort(np.r_[a, b])
    Fa = np.searchsorted(a, grid, side="right") / len(a)
    Fb = np.searchsorted(b, grid, side="right") / len(b)
    return np.abs(Fa - Fb).max()

a = trial.loc[trial["group"] == "control", "minutes_active"].to_numpy()
b = trial.loc[trial["group"] == "gamified", "minutes_active"].to_numpy()
d = ks_distance(a, b)

In [ ]:
xs, ys = ecdf([3, 1, 2, 2])
assert list(xs) == [1, 2, 2, 3] and np.allclose(ys, [0.25, 0.5, 0.75, 1.0])
assert abs(ks_distance([1, 2, 3], [10, 11, 12]) - 1.0) < 1e-12, "completely separated samples have distance 1"
assert abs(d - stats.ks_2samp(a, b).statistic) < 1e-12 and abs(d - 0.24) < 1e-9
print(f"✅ Correct! KS distance = {d:.2f}: at the widest point the two ECDFs are {d:.0%} apart")

## Exercise 3

**Kernel density by hand.** Write `kde_at(x, points, h=None)` returning the Gaussian KDE $\hat f(p)=\frac1{nh}\sum_iK\big((p-x_i)/h\big)$ at each point, with $K(u)=e^{-u^2/2}/\sqrt{2\pi}$ and the default
bandwidth $h=s\,n^{-1/5}$ ($s$ = sample sd with `ddof=1`). Evaluate the exam-score density at 55, 68 and 80 as `dens`.

In [ ]:
def kde_at(x, points, h=None):
    x = np.asarray(x, dtype=float)
    h = h if h is not None else x.std(ddof=1) * len(x) ** (-1 / 5)
    u = (np.asarray(points, dtype=float)[:, None] - x[None, :]) / h
    return (np.exp(-u ** 2 / 2) / math.sqrt(2 * math.pi)).mean(axis=1) / h

scores = students["exam_score"].to_numpy()
dens = kde_at(scores, [55, 68, 80])

In [ ]:
pts = [50.0, 60.0, 70.0, 90.0]
assert np.allclose(kde_at(scores, pts), stats.gaussian_kde(scores)(pts))
assert np.allclose(dens, [0.01844, 0.03774, 0.01715], atol=1e-5), dens
xs = np.linspace(0, 120, 4001)
f = kde_at(scores, xs)
assert abs(((f[:-1] + f[1:]) / 2 * np.diff(xs)).sum() - 1) < 1e-3, "a density integrates to 1"
print("✅ Correct!", dens.round(4))

## Exercise 4

**Box-plot numbers.** Write `box_stats(x)` returning a dict with `q1`, `med`, `q3`, `whislo`, `whishi` and `n_outliers`: quartiles from `np.percentile`, fences $Q_1-1.5\,\text{IQR}$ and $Q_3+1.5\,\text{IQR}$,
whiskers at the most extreme data points **inside** the fences, and the count of points beyond them. Apply it to `response_minutes` as `bs`.

In [ ]:
def box_stats(x):
    x = np.asarray(x, dtype=float)
    q1, med, q3 = np.percentile(x, [25, 50, 75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    inside = x[(x >= lo) & (x <= hi)]
    return {"q1": q1, "med": med, "q3": q3, "whislo": inside.min(), "whishi": inside.max(),
            "n_outliers": int(((x < lo) | (x > hi)).sum())}

bs = box_stats(desk["response_minutes"])

In [ ]:
for data in (students["exam_score"], desk["response_minutes"], trial["minutes_active"]):
    mine, ref = box_stats(data), cbook.boxplot_stats(np.asarray(data), whis=1.5)[0]
    for key in ("q1", "med", "q3", "whislo", "whishi"):
        assert abs(mine[key] - ref[key]) < 1e-9, key
    assert mine["n_outliers"] == len(ref["fliers"])
assert bs["n_outliers"] == 25 and abs(bs["whishi"] - 19.87) < 0.01
print(f"✅ Correct! The whisker ends at {bs['whishi']:.2f} (the largest value inside the fence), and {bs['n_outliers']} points are plotted as outliers")

## Exercise 5

**Which error bar?** Write `error_bar_widths(x)` returning the half-widths `{"sd", "se", "ci95"}` (SD with `ddof=1`; SE $=s/\sqrt n$; 95% CI half-width $=t^*_{n-1}\,\text{SE}$).
Then write `wilson_interval(successes, n)` implementing $\dfrac{\hat p+\frac{z^2}{2n}\pm z\sqrt{\frac{\hat p(1-\hat p)}{n}+\frac{z^2}{4n^2}}}{1+\frac{z^2}{n}}$ with $z=1.96$ (use `stats.norm.ppf(0.975)`),
returning `(low, high)`. Apply them as `widths` (gamified `post_score`) and `wil` (33 conversions out of 100).

In [ ]:
def error_bar_widths(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    sd = x.std(ddof=1)
    se = sd / math.sqrt(n)
    return {"sd": sd, "se": se, "ci95": stats.t.ppf(0.975, n - 1) * se}

def wilson_interval(successes, n):
    z = stats.norm.ppf(0.975)
    p = successes / n
    centre = p + z ** 2 / (2 * n)
    half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2))
    denom = 1 + z ** 2 / n
    return (centre - half) / denom, (centre + half) / denom

widths = error_bar_widths(trial.loc[trial["group"] == "gamified", "post_score"])
wil = wilson_interval(33, 100)

In [ ]:
probe = np.random.default_rng(0).normal(50, 10, 40)
w = error_bar_widths(probe)
assert abs(w["sd"] - probe.std(ddof=1)) < 1e-12 and abs(w["se"] - stats.sem(probe)) < 1e-12
assert abs(w["ci95"] - (stats.t.interval(0.95, 39, loc=0, scale=stats.sem(probe))[1])) < 1e-9
assert w["se"] < w["ci95"] < w["sd"]
assert abs(widths["sd"] - 11.19) < 0.01 and abs(widths["se"] - 1.12) < 0.01 and abs(widths["ci95"] - 2.22) < 0.01
ref = proportion_confint(33, 100, method="wilson")
assert abs(wil[0] - ref[0]) < 1e-6 and abs(wil[1] - ref[1]) < 1e-6
print(f"✅ Correct! SD {widths['sd']:.1f} describes people; SE {widths['se']:.2f} / CI ±{widths['ci95']:.2f} describe the mean. Wilson CI for 33/100: ({wil[0]:.3f}, {wil[1]:.3f})")

## Exercise 6

**Smoothers.** Write `rolling_mean(x, window)` (trailing mean over the last `window` values; the first `window - 1` results are `NaN`) and `ewma(x, alpha)` ($s_0=x_0$, $s_t=\alpha x_t+(1-\alpha)s_{t-1}$).
Apply to the daily rides: `roll7 = rolling_mean(y, 7)` and `smooth = ewma(y, 0.2)`.

In [ ]:
def rolling_mean(x, window):
    x = np.asarray(x, dtype=float)
    out = np.full(len(x), np.nan)
    for t in range(window - 1, len(x)):
        out[t] = x[t - window + 1:t + 1].mean()
    return out

def ewma(x, alpha):
    x = np.asarray(x, dtype=float)
    s = np.empty(len(x))
    s[0] = x[0]
    for t in range(1, len(x)):
        s[t] = alpha * x[t] + (1 - alpha) * s[t - 1]
    return s

y = rides["rides"].to_numpy()
roll7 = rolling_mean(y, 7)
smooth = ewma(y, 0.2)

In [ ]:
series = pd.Series(y)
assert np.isnan(roll7[:6]).all() and not np.isnan(roll7[6])
assert np.allclose(roll7[6:], series.rolling(7).mean().to_numpy()[6:])
assert np.allclose(smooth, series.ewm(alpha=0.2, adjust=False).mean().to_numpy())
assert list(rolling_mean([1, 2, 3, 4], 2)[1:]) == [1.5, 2.5, 3.5]
print("✅ Correct!")

## Exercise 7

**Confidence vs prediction band.** Write `regression_band(x, y, x0, kind="mean")` returning `(fit, low, high)` at the points `x0` for a 95% band: with $s^2=\text{SSE}/(n-2)$, $S_{xx}=\sum(x_i-\bar x)^2$,
leverage term $\frac1n+\frac{(x_0-\bar x)^2}{S_{xx}}$, the half-width is $t^*\,s\sqrt{\text{leverage}}$ for `kind="mean"` and $t^*\,s\sqrt{1+\text{leverage}}$ for `kind="pred"`.
Compute the band for the mean exam score at 10 study hours as `band_mean`, and the prediction band for a new student as `band_pred`.

In [ ]:
def regression_band(x, y, x0, kind="mean"):
    x, y, x0 = (np.asarray(v, dtype=float) for v in (x, y, x0))
    n = len(x)
    b1, b0 = np.polyfit(x, y, 1)
    s = math.sqrt(((y - (b0 + b1 * x)) ** 2).sum() / (n - 2))
    lever = 1 / n + (x0 - x.mean()) ** 2 / ((x - x.mean()) ** 2).sum()
    half = stats.t.ppf(0.975, n - 2) * s * np.sqrt(lever + (1 if kind == "pred" else 0))
    fit = b0 + b1 * x0
    return fit, fit - half, fit + half

xs, ys = students["study_hours_week"], students["exam_score"]
band_mean = regression_band(xs, ys, [10])
band_pred = regression_band(xs, ys, [10], kind="pred")

In [ ]:
grid = np.array([3.0, 10.0, 25.0])
ref = smf.ols("exam_score ~ study_hours_week", students).fit().get_prediction(pd.DataFrame({"study_hours_week": grid})).summary_frame(alpha=0.05)
f, lo, hi = regression_band(xs, ys, grid)
assert np.allclose(lo, ref["mean_ci_lower"]) and np.allclose(hi, ref["mean_ci_upper"])
f, lo, hi = regression_band(xs, ys, grid, kind="pred")
assert np.allclose(lo, ref["obs_ci_lower"]) and np.allclose(hi, ref["obs_ci_upper"])
mean_width = (band_mean[2] - band_mean[1])[0]
pred_width = (band_pred[2] - band_pred[1])[0]
assert pred_width > 10 * mean_width, "A new student's score is far less certain than the average score"
print(f"✅ Correct! At 10 hours the mean score is {band_mean[0][0]:.1f} ± {mean_width / 2:.1f}, but a new student's score is ± {pred_width / 2:.1f}")

## Exercise 8

**A correlation matrix from scratch.** Write `corr_matrix(df, method="pearson")` returning a DataFrame: standardise each column (`ddof=1`) and compute $R=Z^\top Z/(n-1)$; for `method="spearman"` first replace each column by its `rank()`.
Apply it to `cols` as `R` (Pearson) and `Rs` (Spearman).

In [ ]:
def corr_matrix(df, method="pearson"):
    data = df.rank() if method == "spearman" else df
    z = (data - data.mean()) / data.std(ddof=1)
    values = z.to_numpy().T @ z.to_numpy() / (len(df) - 1)
    return pd.DataFrame(values, index=df.columns, columns=df.columns)

cols = ["study_hours_week", "sleep_hours", "attendance_pct", "exam_score"]
R = corr_matrix(students[cols])
Rs = corr_matrix(students[cols], method="spearman")

In [ ]:
assert np.allclose(R, students[cols].corr()) and np.allclose(Rs, students[cols].corr(method="spearman"))
assert list(R.columns) == cols and np.allclose(np.diag(R), 1)
assert abs(R.loc["study_hours_week", "exam_score"] - 0.718) < 0.001
print("✅ Correct!")

## Exercise 9

**2-D binning.** Write `hist2d_counts(x, y, xedges, yedges)` returning an integer matrix of counts (rows = $x$ bins, columns = $y$ bins; a value on an interior edge goes to the bin on its right; the very last edge belongs to the last bin).
Apply it to the non-missing temperature and rides with 8 equal bins in each direction as `counts2d`.

In [ ]:
def hist2d_counts(x, y, xedges, yedges):
    counts = np.zeros((len(xedges) - 1, len(yedges) - 1), dtype=int)
    for xi, yi in zip(x, y):
        i = min(np.searchsorted(xedges, xi, side="right") - 1, len(xedges) - 2)
        j = min(np.searchsorted(yedges, yi, side="right") - 1, len(yedges) - 2)
        counts[i, j] += 1
    return counts

valid = rides.dropna(subset=["temp_c"])
xe = np.linspace(valid["temp_c"].min(), valid["temp_c"].max(), 9)
ye = np.linspace(valid["rides"].min(), valid["rides"].max(), 9)
counts2d = hist2d_counts(valid["temp_c"], valid["rides"], xe, ye)

In [ ]:
ref, _, _ = np.histogram2d(valid["temp_c"], valid["rides"], bins=[xe, ye])
assert counts2d.shape == (8, 8) and (counts2d == ref).all()
assert counts2d.sum() == len(valid) == 360
print("✅ Correct! The busiest cell holds", counts2d.max(), "days")

## Exercise 10

**Read the error bars.** A colleague shows a bar chart of group means with error bars and wants to conclude that the groups differ. What is the best response? Set `answer` to a letter.

* **a)** "Error bars are always standard deviations, so overlapping bars mean no difference."
* **b)** "Which interval is it? SD bars show the spread of individuals, not uncertainty in the means; to compare means, plot 95% confidence intervals and test (or show the CI of) the *difference*."
* **c)** "Non-overlapping standard-error bars prove the difference is real at the 1% level."
* **d)** "Error bars are decoration; the heights of the bars are what matter."

In [ ]:
answer = "b"

In [ ]:
assert answer == "b", "The caption must say what the bars are; SD, SE and CI answer different questions"
print("✅ Correct!")